In [ ]:
import os
import cv2


def load_labels(label_path):
    with open(label_path, "r") as f:
        content = f.read().strip()
    return [ch for ch in content if ch.isdigit()]


def count_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    count = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        count += 1

    cap.release()
    return count


def analyze_video(video_path, label_path):
    labels = load_labels(label_path)
    frame_count = count_video_frames(video_path)
    label_count = len(labels)

    diff = label_count - frame_count

    print("=" * 72)
    print(f"[VIDEO] {video_path}")
    print(f"[LABEL] {label_path}")
    print(f"[INFO] Frames : {frame_count}")
    print(f"[INFO] Labels : {label_count}")

    if diff == 0:
        print(f"[STATUS] PERFECT MATCH ✅")

    elif diff > 0:
        print(f"[STATUS] EXTRA LABELS ⚠️")
        print(f"[INFO] Extra labels : {diff}")

    else:
        print(f"[STATUS] EXTRA FRAMES ⚠️")
        print(f"[INFO] Extra frames : {-diff}")

    print("=" * 72)


def process_root(root_dir):
    for person_id in sorted(os.listdir(root_dir)):
        person_path = os.path.join(root_dir, person_id)
        if not os.path.isdir(person_path):
            continue

        for condition in sorted(os.listdir(person_path)):
            condition_path = os.path.join(person_path, condition)
            if not os.path.isdir(condition_path):
                continue

            for file in sorted(os.listdir(condition_path)):
                if not file.lower().endswith(".avi"):
                    continue

                video_name = os.path.splitext(file)[0]
                video_path = os.path.join(condition_path, file)

                label_path = os.path.join(
                    condition_path,
                    f"{person_id}_{video_name}_head.txt"
                )

                if not os.path.exists(label_path):
                    print(f"[WARNING] Missing label: {label_path}")
                    continue

                analyze_video(video_path, label_path)


ROOT_DIR = "/root/.cache/kagglehub/datasets/manith04/ddewewewee/versions/1/"

if __name__ == "__main__":
    process_root(ROOT_DIR)